# Main Orchestrator — user perspective

**What it does.** Top-level dispatcher. Calls the Global Orchestrator, then routes between two modes:

- **cascade** — runs the first ready task synchronously in the main working tree (Phase 3 behaviour, no claiming).
- **parallel** — sequential optimistic-lock claiming (MORD-03), per-task `git worktree add` (MORD-04), `asyncio.gather` over WIO subprocesses, finally posts a structured cycle summary to the EPIC (MORD-05).

**Pure Python.** No LLM, no SDK. Just dispatch + worktree management.

**Entry point.** `await run_main_orchestrator(mode="cascade" | "parallel")`.

**Cost.** Live runs spawn real WIO subprocesses, create real branches and worktrees, and post a real Linear comment. Heavily gated.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Pure-logic probe — `_build_cycle_summary`

The cycle-summary builder formats the Linear comment posted at the end of a parallel run. Runs offline.

In [ ]:
from hsb.agents.main_orchestrator import _build_cycle_summary
from hsb.contracts.main_orchestrator import DispatchedItem

dispatched = [
    DispatchedItem(
        work_item_id="LIN-1",
        orchestrator_instance="parallel-0",
        claim_status="claimed",
        final_status="completed",
    ),
    DispatchedItem(
        work_item_id="LIN-2",
        orchestrator_instance="parallel-1",
        claim_status="claimed",
        final_status="failed",
    ),
    DispatchedItem(
        work_item_id="LIN-3",
        orchestrator_instance="skipped",
        claim_status="skipped",
        final_status="blocked",
    ),
]
print(_build_cycle_summary(mode="parallel", dispatched=dispatched))

## Live invocation (heavily gated)

Spawns WIO subprocesses, creates `.worktrees/LIN-*` directories, posts a Linear summary. Run only with a sandbox project. Notebook 03 has more granular probes for the dispatch internals.

In [ ]:
from _helpers import assert_g1_safe, gated, live_mode

if not live_mode():
    print(gated("Main Orchestrator live run — spawns WIO subprocesses"))
else:
    assert_g1_safe()
    import asyncio

    from hsb.agents.main_orchestrator import run_main_orchestrator

    asyncio.run(run_main_orchestrator(mode="cascade"))
    print("cycle complete — see Linear EPIC for summary comment")